# RAG : Usando Pinecone como base de conocimiento

In [4]:
from pinecone import Pinecone, PodSpec, ServerlessSpec

import os, getpass

In [3]:
os.environ["PINECONE_API_KEY"] = getpass.getpass("Ingresa tu API Key de Pinecone : ")
api_key = os.environ["PINECONE_API_KEY"]

## Creando un index en Pinecone

In [5]:
index_name = "knowledge-base-coctel"
dimension = 1536 #Dimensiones del modelo embedding de OpenAI

pc = Pinecone(api_key=api_key)

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print(f"Index {index_name} creado con éxito en Pinecone")

Index knowledge-base-coctel creado con éxito en Pinecone


## Cargando y generando fragmentos de un archivo PDF

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

loader = PyPDFLoader("pdf/1000coctelesdetodoelmundo.pdf")
data = loader.load()

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 400, 
    chunk_overlap = 20
)

docs = text_splitter.split_documents(documents = data)

In [15]:
len(docs), docs

(622,
 [Document(metadata={'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2023-10-30T20:39:56+00:00', 'title': '1000 cócteles de todo el mundo', 'author': 'Antonio Primiceri', 'moddate': '2023-10-30T20:40:34+00:00', 'source': 'pdf/1000coctelesdetodoelmundo.pdf', 'total_pages': 628, 'page': 3, 'page_label': '4'}, page_content='Antonio Primiceri\ncon la colaboración de Roberto Savioli'),
  Document(metadata={'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2023-10-30T20:39:56+00:00', 'title': '1000 cócteles de todo el mundo', 'author': 'Antonio Primiceri', 'moddate': '2023-10-30T20:40:34+00:00', 'source': 'pdf/1000coctelesdetodoelmundo.pdf', 'total_pages': 628, 'page': 4, 'page_label': '5'}, page_content='A pesar de haber puesto el máximo cuidado en la redacción de esta obra, el autor o el editor no pueden\nen modo alguno responsabilizarse por las informaciones (fórmulas, recetas, técnicas, etc.) vertidas en el\ntexto. Se aconseja, en el caso de problemas específicos 

In [8]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu API Key de OpenAI : ")

## Cargando datos a Pinecone

In [18]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

template = """Responde a la pregunta basándote solo en el siguiente contexto.
Si no puedes responder a la pregunta, responde exactamente:
"No lo sé disculpa, puedes buscar la información en internet."

<context>
{context}
</context>

Pregunta: {input}
Respuesta:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "input"]
)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0
)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)

qa = create_retrieval_chain(
    vectorstore.as_retriever(search_kwargs={"k": 4}),
    combine_docs_chain
)

respuesta = qa.invoke({"input": "¿De qué trata el documento?"})
print(respuesta["answer"])

No lo sé disculpa, puedes buscar la información en internet.


In [22]:
qa.invoke({"input": "¿puedes recomendarme cócteles refrescantes? y por favor dame la receta para 2 personas"})["answer"]

'Agite en la coctelera los ingredientes con algunos cubitos de hielo. Sirva en copas de cóctel y termine de llenarlas con tónica. Adorne con kiwi y fresas.'

## Eliminando index en Pinecone

In [23]:
pc.delete_index(index_name)